# 04 - Exportación de artefactos para demo

Este notebook junta los outputs del ranker y de embeddings, y arma una carpeta ligera para la app de Streamlit.

La idea es no subir todo el dataset ni todos los archivos pesados al repositorio. En su lugar se crea una muestra suficiente para demostrar el funcionamiento del pipeline.


## Revisión de inputs disponibles

Primero imprimo todos los archivos montados en `/kaggle/input`. Esto sirve para confirmar que agregué como input los outputs de los notebooks 01, 02 y 03.


In [2]:
import os
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Archivos disponibles en /kaggle/input:\n")

for dirname, _, filenames in os.walk(INPUT):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Archivos disponibles en /kaggle/input:

/kaggle/input/notebooks/samanthacarolina/02-train-ranker/ranking_metrics.json
/kaggle/input/notebooks/samanthacarolina/02-train-ranker/ltr_test.parquet
/kaggle/input/notebooks/samanthacarolina/02-train-ranker/__results__.html
/kaggle/input/notebooks/samanthacarolina/02-train-ranker/ltr_train.parquet
/kaggle/input/notebooks/samanthacarolina/02-train-ranker/__notebook__.ipynb
/kaggle/input/notebooks/samanthacarolina/02-train-ranker/__output__.json
/kaggle/input/notebooks/samanthacarolina/02-train-ranker/xgb_ranker.joblib
/kaggle/input/notebooks/samanthacarolina/02-train-ranker/custom.css
/kaggle/input/notebooks/samanthacarolina/01-prepare-data/__results__.html
/kaggle/input/notebooks/samanthacarolina/01-prepare-data/product_catalog.parquet
/kaggle/input/notebooks/samanthacarolina/01-prepare-data/__notebook__.ipynb
/kaggle/input/notebooks/samanthacarolina/01-prepare-data/orders.parquet
/kaggle/input/notebooks/samanthacarolina/01-prepare-data/__outpu

## Búsqueda automática de archivos

Defino una función pequeña para encontrar cada archivo por nombre. Así evito depender de nombres exactos de carpetas generadas por Kaggle.


In [3]:
def find_file(filename: str) -> Path:
    matches = list(INPUT.rglob(filename))
    
    if not matches:
        raise FileNotFoundError(f"No se encontró {filename} en /kaggle/input")
    
    print(f"{filename} -> {matches[0]}")
    return matches[0]


prior_items_path = find_file("prior_items.parquet")
product_catalog_path = find_file("product_catalog.parquet")

ranker_path = find_file("xgb_ranker.joblib")
ranking_metrics_path = find_file("ranking_metrics.json")

text_embeddings_path = find_file("text_embeddings.npy")
text_product_ids_path = find_file("text_product_ids.npy")
product_catalog_embeddings_path = find_file("product_catalog_embeddings.parquet")

embedding_report_path = find_file("embedding_report.json")
embedding_latency_path = find_file("embedding_latency.json")
embedding_examples_path = find_file("embedding_examples.json")

prior_items.parquet -> /kaggle/input/notebooks/samanthacarolina/01-prepare-data/prior_items.parquet
product_catalog.parquet -> /kaggle/input/notebooks/samanthacarolina/01-prepare-data/product_catalog.parquet
xgb_ranker.joblib -> /kaggle/input/notebooks/samanthacarolina/02-train-ranker/xgb_ranker.joblib
ranking_metrics.json -> /kaggle/input/notebooks/samanthacarolina/02-train-ranker/ranking_metrics.json
text_embeddings.npy -> /kaggle/input/notebooks/samanthacarolina/03-train-embeddings/text_embeddings.npy
text_product_ids.npy -> /kaggle/input/notebooks/samanthacarolina/03-train-embeddings/text_product_ids.npy
product_catalog_embeddings.parquet -> /kaggle/input/notebooks/samanthacarolina/03-train-embeddings/product_catalog_embeddings.parquet
embedding_report.json -> /kaggle/input/notebooks/samanthacarolina/03-train-embeddings/embedding_report.json
embedding_latency.json -> /kaggle/input/notebooks/samanthacarolina/03-train-embeddings/embedding_latency.json
embedding_examples.json -> /kagg

## Carga de datos base

Cargo el historial, el catálogo y los artefactos de embeddings. Estos datos se usarán para crear una versión reducida de la demo.


In [4]:
prior_items = pd.read_parquet(prior_items_path)
product_catalog = pd.read_parquet(product_catalog_path)
product_catalog_embeddings = pd.read_parquet(product_catalog_embeddings_path)

text_embeddings = np.load(text_embeddings_path)
text_product_ids = np.load(text_product_ids_path)

print("prior_items:", prior_items.shape)
print("product_catalog:", product_catalog.shape)
print("product_catalog_embeddings:", product_catalog_embeddings.shape)
print("text_embeddings:", text_embeddings.shape)
print("text_product_ids:", text_product_ids.shape)

prior_items: (32434489, 14)
product_catalog: (49688, 6)
product_catalog_embeddings: (49688, 7)
text_embeddings: (49688, 384)
text_product_ids: (49688,)


## Carga de reportes

Cargo los reportes de ranking y embeddings para incluirlos en la carpeta final y mostrarlos en Streamlit.


In [5]:
with open(ranking_metrics_path, "r") as f:
    ranking_metrics = json.load(f)

with open(embedding_report_path, "r") as f:
    embedding_report = json.load(f)

with open(embedding_latency_path, "r") as f:
    embedding_latency = json.load(f)

with open(embedding_examples_path, "r") as f:
    embedding_examples = json.load(f)

print("Ranking metrics:")
print(json.dumps(ranking_metrics, indent=2)[:1000])

print("\nEmbedding latency:")
print(json.dumps(embedding_latency, indent=2))

Ranking metrics:
{
  "sample_users": 5000,
  "negative_sampling": {
    "train_negatives_per_positive": 5,
    "test_negatives_per_positive": 20,
    "strategy": "same_department_products_never_bought_by_user"
  },
  "features": [
    "user_product_frequency",
    "user_product_reordered",
    "product_global_popularity",
    "department_id"
  ],
  "baseline": {
    "ndcg_at_10": 0.5031041751533476,
    "map_at_10": 0.3679661195515243,
    "num_queries": 5000
  },
  "xgb_ranker": {
    "ndcg_at_10": 0.9187133675832192,
    "map_at_10": 0.8016026298500881,
    "num_queries": 5000
  },
  "feature_importance": [
    {
      "feature": "user_product_reordered",
      "importance": 0.5025421977043152
    },
    {
      "feature": "user_product_frequency",
      "importance": 0.48458120226860046
    },
    {
      "feature": "product_global_popularity",
      "importance": 0.011094234883785248
    },
    {
      "feature": "department_id",
      "importance": 0.0017824183451011777
    }
  ]


## Carpeta final de artefactos

Creo una estructura limpia con subcarpetas para ranking, embeddings, datos y reportes.


In [6]:
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts_sample"

if ARTIFACTS_DIR.exists():
    shutil.rmtree(ARTIFACTS_DIR)

(ARTIFACTS_DIR / "ranking").mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / "embeddings").mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / "data").mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / "reports").mkdir(parents=True, exist_ok=True)

print("Carpeta creada:")
print(ARTIFACTS_DIR)

Carpeta creada:
/kaggle/working/artifacts_sample


## Modelo de ranking y métricas

Copio el modelo entrenado y el reporte de métricas de ranking al paquete final.


In [7]:
shutil.copy(ranker_path, ARTIFACTS_DIR / "ranking" / "xgb_ranker.joblib")
shutil.copy(ranking_metrics_path, ARTIFACTS_DIR / "reports" / "ranking_metrics.json")

print("Copiado:")
print("- ranking/xgb_ranker.joblib")
print("- reports/ranking_metrics.json")

Copiado:
- ranking/xgb_ranker.joblib
- reports/ranking_metrics.json


## Muestra de usuarios para demo

Selecciono una muestra de usuarios con historial. La app usará estos usuarios para que el evaluador pueda probar recomendaciones sin cargar todo el dataset.


In [8]:
DEMO_USERS = 1000

demo_user_ids = (
    prior_items["user_id"]
    .drop_duplicates()
    .sample(
        n=min(DEMO_USERS, prior_items["user_id"].nunique()),
        random_state=42
    )
)

user_history_sample = prior_items[
    prior_items["user_id"].isin(demo_user_ids)
].copy()

print("Usuarios demo:", user_history_sample["user_id"].nunique())
print("Filas historial demo:", user_history_sample.shape)

user_history_sample.head()

Usuarios demo: 1000
Filas historial demo: (150405, 14)


,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,eval_set,order_dow,order_hour_of_day,product_name,aisle_id,department_id,aisle,department
977,109,36217,1,1,86865,20,prior,0,9,"Dressing & Marinade, Toasted Sesame",89,13,salad dressing toppings,pantry
978,109,8195,2,1,86865,20,prior,0,9,Paper Towels,54,17,paper goods,household
979,109,25146,3,1,86865,20,prior,0,9,Original Orange Juice,31,7,refrigerated,beverages
980,109,7948,4,0,86865,20,prior,0,9,Organic Unsalted Butter,36,16,butter,dairy eggs
981,109,4715,5,1,86865,20,prior,0,9,"Kitchen Cleaner + Bleach, Fresh Scent",114,17,cleaning products,household


## Popularidad global

Calculo la popularidad global de productos. La uso como señal simple en la demo y también como feature para el ranker.


In [9]:
product_popularity = (
    prior_items
    .groupby("product_id")["user_id"]
    .nunique()
    .reset_index(name="product_global_popularity")
)

product_popularity = product_popularity.sort_values(
    "product_global_popularity",
    ascending=False
)

product_popularity.head()

,product_id,product_global_popularity
24848,24852,73956
13172,13176,63537
21133,21137,58838
21899,21903,55037
47615,47626,46402


## Datos livianos para Streamlit

Guardo solo los archivos necesarios para que la app funcione: historial sampleado, catálogo y popularidad.


In [10]:
user_history_sample.to_parquet(
    ARTIFACTS_DIR / "data" / "user_history_sample.parquet",
    index=False
)

product_catalog.to_parquet(
    ARTIFACTS_DIR / "data" / "product_catalog.parquet",
    index=False
)

product_popularity.to_parquet(
    ARTIFACTS_DIR / "data" / "product_popularity.parquet",
    index=False
)

print("Guardado:")
print("- data/user_history_sample.parquet")
print("- data/product_catalog.parquet")
print("- data/product_popularity.parquet")

Guardado:
- data/user_history_sample.parquet
- data/product_catalog.parquet
- data/product_popularity.parquet


## Reducción de embeddings

Para que el repo y la app no sean demasiado pesados, reduzco el índice a los productos más populares.


In [11]:
MAX_DEMO_PRODUCTS = 15000

top_product_ids = (
    product_popularity
    .head(MAX_DEMO_PRODUCTS)["product_id"]
    .astype(int)
    .tolist()
)

top_product_ids_set = set(top_product_ids)

mask = np.array([
    int(pid) in top_product_ids_set
    for pid in text_product_ids
])

demo_text_embeddings = text_embeddings[mask]
demo_text_product_ids = text_product_ids[mask]

print("Embeddings completos:", text_embeddings.shape)
print("Embeddings demo:", demo_text_embeddings.shape)
print("Product ids demo:", demo_text_product_ids.shape)

Embeddings completos: (49688, 384)
Embeddings demo: (15000, 384)
Product ids demo: (15000,)


## Instalación de FAISS

En Kaggle cada notebook tiene su propio entorno, así que dejo explícita la instalación de FAISS antes de crear el índice reducido.


In [12]:
import sys

!{sys.executable} -m pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.9 MB/s eta 0:00:00


## Índice FAISS reducido

Creo un índice vectorial más liviano para la demo y guardo los vectores junto con sus product IDs.


In [13]:
import faiss

demo_text_embeddings = demo_text_embeddings.astype("float32")

demo_faiss_index = faiss.IndexFlatIP(demo_text_embeddings.shape[1])
demo_faiss_index.add(demo_text_embeddings)

faiss.write_index(
    demo_faiss_index,
    str(ARTIFACTS_DIR / "embeddings" / "text_products_demo.faiss")
)

np.save(
    ARTIFACTS_DIR / "embeddings" / "text_embeddings_demo.npy",
    demo_text_embeddings
)

np.save(
    ARTIFACTS_DIR / "embeddings" / "text_product_ids_demo.npy",
    demo_text_product_ids
)

print("Índice FAISS demo creado")
print("Vectores:", demo_faiss_index.ntotal)

Índice FAISS demo creado
Vectores: 15000


## Catálogo alineado al índice

Guardo solo el catálogo de los productos incluidos en el índice reducido.


In [14]:
product_catalog_embeddings_demo = product_catalog_embeddings[
    product_catalog_embeddings["product_id"].isin(demo_text_product_ids)
].copy()

product_catalog_embeddings_demo.to_parquet(
    ARTIFACTS_DIR / "embeddings" / "product_catalog_embeddings_demo.parquet",
    index=False
)

print("Catálogo embeddings demo:", product_catalog_embeddings_demo.shape)

Catálogo embeddings demo: (15000, 7)


## Reportes de embeddings

Copio los reportes y ejemplos de embeddings para que la app pueda mostrarlos sin recalcular nada.


In [15]:
shutil.copy(
    embedding_report_path,
    ARTIFACTS_DIR / "reports" / "embedding_report.json"
)

shutil.copy(
    embedding_latency_path,
    ARTIFACTS_DIR / "reports" / "embedding_latency.json"
)

shutil.copy(
    embedding_examples_path,
    ARTIFACTS_DIR / "reports" / "embedding_examples.json"
)

print("Reportes copiados")

Reportes copiados


## Manifest del paquete

Creo un archivo `manifest.json` para documentar qué contiene cada carpeta del paquete final.


In [16]:
manifest = {
    "description": "Sample artifacts for Streamlit demo",
    "created_from_notebooks": [
        "01_prepare_data",
        "02_train_ranker",
        "03_train_embeddings"
    ],
    "ranking": {
        "model": "ranking/xgb_ranker.joblib",
        "metrics": "reports/ranking_metrics.json"
    },
    "embeddings": {
        "faiss_index": "embeddings/text_products_demo.faiss",
        "text_embeddings": "embeddings/text_embeddings_demo.npy",
        "product_ids": "embeddings/text_product_ids_demo.npy",
        "catalog": "embeddings/product_catalog_embeddings_demo.parquet",
        "max_demo_products": MAX_DEMO_PRODUCTS
    },
    "data": {
        "user_history_sample": "data/user_history_sample.parquet",
        "product_catalog": "data/product_catalog.parquet",
        "product_popularity": "data/product_popularity.parquet",
        "demo_users": DEMO_USERS
    },
    "reports": {
        "ranking_metrics": "reports/ranking_metrics.json",
        "embedding_report": "reports/embedding_report.json",
        "embedding_latency": "reports/embedding_latency.json",
        "embedding_examples": "reports/embedding_examples.json"
    }
}

with open(ARTIFACTS_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))

{
  "description": "Sample artifacts for Streamlit demo",
  "created_from_notebooks": [
    "01_prepare_data",
    "02_train_ranker",
    "03_train_embeddings"
  ],
  "ranking": {
    "model": "ranking/xgb_ranker.joblib",
    "metrics": "reports/ranking_metrics.json"
  },
  "embeddings": {
    "faiss_index": "embeddings/text_products_demo.faiss",
    "text_embeddings": "embeddings/text_embeddings_demo.npy",
    "product_ids": "embeddings/text_product_ids_demo.npy",
    "catalog": "embeddings/product_catalog_embeddings_demo.parquet",
    "max_demo_products": 15000
  },
  "data": {
    "user_history_sample": "data/user_history_sample.parquet",
    "product_catalog": "data/product_catalog.parquet",
    "product_popularity": "data/product_popularity.parquet",
    "demo_users": 1000
  },
  "reports": {
    "ranking_metrics": "reports/ranking_metrics.json",
    "embedding_report": "reports/embedding_report.json",
    "embedding_latency": "reports/embedding_latency.json",
    "embedding_examp

## README interno

Agrego un README dentro de `artifacts_sample` para que sea claro qué archivos son de demo y de dónde vienen.


In [17]:
readme_text = f"""# Artifacts Sample

This folder contains lightweight artifacts for the Streamlit demo.

## Contents

### Ranking
- `ranking/xgb_ranker.joblib`: trained XGBRanker model.
- `reports/ranking_metrics.json`: NDCG@10 and MAP@10 results.

### Embeddings
- `embeddings/text_products_demo.faiss`: FAISS index for semantic product search.
- `embeddings/text_embeddings_demo.npy`: normalized text embeddings.
- `embeddings/text_product_ids_demo.npy`: product IDs aligned with the embeddings.
- `embeddings/product_catalog_embeddings_demo.parquet`: product metadata for the demo index.

### Data
- `data/user_history_sample.parquet`: sampled user purchase history for demo users.
- `data/product_catalog.parquet`: product catalog with names, aisles and departments.
- `data/product_popularity.parquet`: product global popularity.

## Notes

The full training was executed in Kaggle notebooks.  
This folder is reduced to keep the GitHub repository and Streamlit demo lightweight.

Demo users: {DEMO_USERS}  
Demo products in semantic index: {MAX_DEMO_PRODUCTS}
"""

with open(ARTIFACTS_DIR / "README.md", "w") as f:
    f.write(readme_text)

print(readme_text)

# Artifacts Sample

This folder contains lightweight artifacts for the Streamlit demo.

## Contents

### Ranking
- `ranking/xgb_ranker.joblib`: trained XGBRanker model.
- `reports/ranking_metrics.json`: NDCG@10 and MAP@10 results.

### Embeddings
- `embeddings/text_products_demo.faiss`: FAISS index for semantic product search.
- `embeddings/text_embeddings_demo.npy`: normalized text embeddings.
- `embeddings/text_product_ids_demo.npy`: product IDs aligned with the embeddings.
- `embeddings/product_catalog_embeddings_demo.parquet`: product metadata for the demo index.

### Data
- `data/user_history_sample.parquet`: sampled user purchase history for demo users.
- `data/product_catalog.parquet`: product catalog with names, aisles and departments.
- `data/product_popularity.parquet`: product global popularity.

## Notes

The full training was executed in Kaggle notebooks.  
This folder is reduced to keep the GitHub repository and Streamlit demo lightweight.

Demo users: 1000  
Demo product

## Compresión final

Genero `artifacts_sample.zip`, que es el archivo que luego se descarga y se sube al repositorio.


In [18]:
zip_path = shutil.make_archive(
    str(OUTPUT_DIR / "artifacts_sample"),
    "zip",
    ARTIFACTS_DIR
)

print("ZIP creado:")
print(zip_path)

ZIP creado:
/kaggle/working/artifacts_sample.zip


## Tamaño de archivos

Reviso el peso del paquete final. Si queda muy grande, se puede bajar `MAX_DEMO_PRODUCTS` o `DEMO_USERS`.


In [19]:
def get_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 * 1024)


print("Tamaños de archivos:\n")

for path in sorted(ARTIFACTS_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(ARTIFACTS_DIR)}: {get_size_mb(path):.2f} MB")

zip_file = OUTPUT_DIR / "artifacts_sample.zip"
print(f"\nZIP total: {get_size_mb(zip_file):.2f} MB")

Tamaños de archivos:

README.md: 0.00 MB
data/product_catalog.parquet: 1.49 MB
data/product_popularity.parquet: 0.32 MB
data/user_history_sample.parquet: 1.73 MB
embeddings/product_catalog_embeddings_demo.parquet: 0.83 MB
embeddings/text_embeddings_demo.npy: 21.97 MB
embeddings/text_product_ids_demo.npy: 0.11 MB
embeddings/text_products_demo.faiss: 21.97 MB
manifest.json: 0.00 MB
ranking/xgb_ranker.joblib: 0.68 MB
reports/embedding_examples.json: 0.01 MB
reports/embedding_latency.json: 0.00 MB
reports/embedding_report.json: 0.00 MB
reports/ranking_metrics.json: 0.00 MB

ZIP total: 44.52 MB


## Verificación final

Confirmo que la carpeta y el ZIP final quedaron en `/kaggle/working`.


In [20]:
print("Archivos finales en /kaggle/working:\n")

for file in sorted(os.listdir("/kaggle/working")):
    print("-", file)

Archivos finales en /kaggle/working:

- __notebook__.ipynb
- artifacts_sample
- artifacts_sample.zip
